In [8]:
import os
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision.models import mobilenet_v3_large, MobileNet_V3_Large_Weights
import torchvision.transforms as T
from PIL import Image
from tqdm import tqdm

In [ ]:
class KonIQDataset(Dataset):
    def __init__(self, csv_file, img_dir, transform=None):
        self.data = pd.read_csv(csv_file) 
        self.img_dir = img_dir
        self.transform = transform

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        img_name = self.data.iloc[idx]['image_name']
        mos = float(self.data.iloc[idx]['MOS'])
        
        img_path = os.path.join(self.img_dir, img_name)
        image = Image.open(img_path).convert('RGB')
        
        if self.transform:
            image = self.transform(image)
            
        target_normalized = (mos - 1.0) / 4.0 
        
        return image, torch.tensor([target_normalized], dtype=torch.float32)


In [ ]:
# final layer, make it for regression
class IQAProxy(nn.Module):
    def __init__(self):
        super().__init__()
        self.model = mobilenet_v3_large(weights=MobileNet_V3_Large_Weights.IMAGENET1K_V2)
        
        self.model.classifier = nn.Sequential(
            nn.Linear(960, 1280),
            nn.Hardswish(),
            nn.Dropout(0.2),
            nn.Linear(1280, 1),
            nn.Sigmoid() 
        )

    def forward(self, x):
        return self.model(x)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Training on: {device}")

CSV_PATH = r"dataset\konIQ-10K_scores\koniq10k_scores_and_distributions.csv"  
IMG_DIR = r"dataset\konIQ-10K\512x384"

transform = T.Compose([
        T.RandomCrop((256, 256)), 
        T.RandomHorizontalFlip(p=0.5),
        T.RandomVerticalFlip(p=0.2),
        T.ToTensor(),
        # for imagenet based models, we normalize this way
        T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

full_dataset = KonIQDataset(csv_file=CSV_PATH, img_dir=IMG_DIR, transform=transform)
train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size
train_dataset, val_dataset = random_split(full_dataset, [train_size, val_size])
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=0, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=0, pin_memory=True)

proxy_net = IQAProxy().to(device)
optimizer = optim.Adam(proxy_net.parameters(), lr=1e-5)
criterion = nn.MSELoss()

Training on: cuda


In [ ]:
epochs = 30
best_val_loss = float('inf')

for epoch in range(epochs):
        proxy_net.train()
        running_loss = 0.0
        
        print(f"\nEpoch {epoch+1}/{epochs}")
        train_bar = tqdm(train_loader, desc="Training")
        
        for images, targets in train_bar:
            images, targets = images.to(device), targets.to(device)
            
            optimizer.zero_grad()
            outputs = proxy_net(images)
            loss = criterion(outputs, targets)
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item()
            train_bar.set_postfix({'MSE': f"{loss.item():.4f}"})
            
        avg_train_loss = running_loss / len(train_loader)

        # validation
        proxy_net.eval()
        val_loss = 0.0
        with torch.no_grad():
            for images, targets in val_loader:
                images, targets = images.to(device), targets.to(device)
                outputs = proxy_net(images)
                loss = criterion(outputs, targets)
                val_loss += loss.item()
                
        avg_val_loss = val_loss / len(val_loader)
        print(f"Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f}")

        # Save the best model weights
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            torch.save(proxy_net.state_dict(), "best_IQA_proxy.pth")
            print("--> Saved new best model!")


Epoch 1/30


Training: 100%|██████████| 252/252 [00:28<00:00,  8.76it/s, MSE=0.0083]


Train Loss: 0.0126 | Val Loss: 0.0088
--> Saved new best model!

Epoch 2/30


Training: 100%|██████████| 252/252 [00:28<00:00,  8.95it/s, MSE=0.0060]


Train Loss: 0.0066 | Val Loss: 0.0059
--> Saved new best model!

Epoch 3/30


...

Train Loss: 0.0024 | Val Loss: 0.0036

Epoch 29/30


Training: 100%|██████████| 252/252 [00:28<00:00,  8.86it/s, MSE=0.0022]


Train Loss: 0.0025 | Val Loss: 0.0033
--> Saved new best model!

Epoch 30/30


Training: 100%|██████████| 252/252 [00:28<00:00,  8.85it/s, MSE=0.0016]


Train Loss: 0.0024 | Val Loss: 0.0035
